# Car Detection for Self-Driving Cars — YOLO

I built this notebook while working through the Convolutional Neural Networks course in Andrew Ng's Deep Learning Specialization on DeepLearning.AI. All credit for the YOLO algorithm, the model architecture, and the pre-trained weights goes to the original researchers and to DeepLearning.AI, whose course walked me through the theory behind this pipeline — what's here is my own implementation of the object-detection logic (score filtering, IoU, non-max suppression) applied to a dataset of dashcam-style images.

**Goal:** take the raw output of a YOLO model and turn it into a clean set of labeled bounding boxes around cars (and other objects) in a driving scene.

To get there, this notebook:
- Filters out low-confidence box predictions using a class-score threshold
- Implements Intersection over Union (IoU) to measure how much two boxes overlap
- Uses IoU inside a non-max suppression step to collapse duplicate detections into one box per object
- Loads a pre-trained YOLO model and runs the full pipeline on a real image

Full reference list (papers, weights source, dataset license) is at the bottom.


## Contents

- [Setup](#0)
- [1 - The Problem](#1)
- [2 - How YOLO Works](#2)
    - [2.1 - Model Output Shape](#2-1)
    - [2.2 - Score Thresholding](#2-2)
    - [2.3 - Non-Max Suppression](#2-3)
    - [2.4 - Applying NMS](#2-4)
    - [2.5 - Putting It Together](#2-5)
- [3 - Running the Pre-trained Model](#3)
    - [3.1 - Classes, Anchors, Image Shape](#3-1)
    - [3.2 - Loading the Model](#3-2)
    - [3.3 - Formatting the Model Output](#3-3)
    - [3.4 - Filtering the Boxes](#3-4)
    - [3.5 - Running Inference](#3-5)
- [4 - Recap](#4)
- [5 - References](#5)

<a name='0'></a>
## Setup

Load the libraries and helper utilities used throughout the notebook.

In [ ]:
import numpy as np
import tensorflow as tf
from yad2k.utils.utils import scale_boxes

In [ ]:
import argparse
import os
import matplotlib.pyplot as plt
from matplotlib.pyplot import imshow
import scipy.io
import scipy.misc
import pandas as pd
import PIL
from PIL import ImageFont, ImageDraw, Image

from tensorflow.python.framework.ops import EagerTensor
from tensorflow.keras.models import load_model
from yad2k.models.keras_yolo import yolo_head
from yad2k.utils.utils import draw_boxes, get_colors_for_classes, read_classes, read_anchors, preprocess_image

%matplotlib inline

<a name='1'></a>
## 1 - The Problem

Imagine a camera mounted on the front of a car, snapping a frame of the road every couple of seconds while driving. Each frame needs cars (and other relevant objects) picked out automatically, so a self-driving system knows what's around it.

The images in this dataset come with ground-truth bounding boxes already drawn around every car — that's what a supervised detector is trained against. *(The original notebook embeds a short clip and a labeled example image here; I've left those out since the media files aren't included in this repo.)*

Classes can be represented either as an integer (1–80) or as an 80-dimensional one-hot vector — both representations show up at different points below depending on which is more convenient.

The rest of this section builds up how YOLO turns a raw image into a set of detected objects.

<a name='2'></a>
## 2 - How YOLO Works

YOLO ("You Only Look Once") is popular because it's both accurate and fast enough to run in real time — it gets there by making all of its predictions in a single forward pass through the network, rather than scanning the image region by region. Non-max suppression is applied afterward to clean up the raw predictions into a final set of boxes.

<a name='2-1'></a>
### 2.1 - Model Output Shape

**Input / output**
- Input: a batch of images, each shaped (608, 608, 3)
- Output: a list of bounding boxes, each described by 6 numbers — $(p_c, b_x, b_y, b_h, b_w, c)$ — where $c$ can be expanded into an 80-dim one-hot class vector, giving 85 numbers per box overall.

**Anchor boxes**
* 5 anchor boxes are used here, pre-selected to cover reasonable width/height ratios across the 80 target classes (stored in `model_data/yolo_anchors.txt`).
* Combined with the grid, the encoding tensor has shape $(m, n_H, n_W, anchors, classes)$.
* End to end: an image of shape (m, 608, 608, 3) goes through the CNN and comes out as an encoding of shape (m, 19, 19, 5, 85).

Whichever grid cell contains the center of an object is the cell responsible for detecting it. *(Diagram of the encoding layout omitted — see the YOLO paper for the original figure.)*

With 5 anchor boxes per cell across a 19×19 grid, the last two dimensions of the (19, 19, 5, 85) encoding get flattened for convenience, giving a (19, 19, 425) output overall.

#### Class score

For each of the 5 boxes in every cell, multiplying the "is there an object here" probability by each class probability gives a class score: $score_{c,i} = p_{c} \times c_{i}$.

**Worked example:** say box 1 has a 60% chance of containing *some* object ($p_1 = 0.60$), and given that, a 73% chance it's specifically a car ($c_3 = 0.73$). The score for "car" on that box is $0.60 \times 0.73 = 0.44$. Doing this across all 80 classes and keeping the max gives that box's final predicted class and score.

One way to visualize this: for every grid cell, take the max class-probability across all 5 anchor boxes, and color the cell by whichever class won. This isn't part of the actual detection pipeline — it's just a handy way to sanity-check what the model is "seeing" in an intermediate step.

#### Bounding boxes & non-max suppression

Plotting the raw predicted boxes (even after keeping only high-confidence ones) still leaves way too many — a 19×19×5 grid means up to 1805 candidate boxes from a single forward pass, often several boxes converging on the same object.

**Non-max suppression** is the fix. Two things happen:
- Low-confidence boxes get dropped (score too low, whether because "is there an object" was low or the specific class probability was low).
- When multiple boxes overlap and are clearly detecting the same object, only the highest-scoring one is kept.

<a name='2-2'></a>
### 2.2 - Score Thresholding

The first filtering step: get rid of any box whose best class score falls below a chosen threshold.

The model's raw output is 19×19×5×85 numbers. To make this easier to work with, it helps to split it into:
- `box_confidence`: shape $(19, 19, 5, 1)$ — $p_c$, the model's confidence that *something* is in each of the 5 boxes per cell
- `boxes`: shape $(19, 19, 5, 4)$ — the box coordinates $(b_x, b_y, b_h, b_w)$
- `box_class_probs`: shape $(19, 19, 5, 80)$ — per-class probability, conditional on an object being present

`yolo_filter_boxes` below combines these into a class score per box, thresholds on it, and returns only the boxes that survive.

In [ ]:
def yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = .6):
    """Keeps only the boxes whose top class score clears a given threshold.

    Arguments:
        boxes -- tensor of shape (19, 19, 5, 4)
        box_confidence -- tensor of shape (19, 19, 5, 1)
        box_class_probs -- tensor of shape (19, 19, 5, 80)
        threshold -- boxes whose best class score is below this get dropped

    Returns:
        scores -- tensor of shape (None,), best class score per surviving box
        boxes -- tensor of shape (None, 4), coordinates of surviving boxes
        classes -- tensor of shape (None,), predicted class index per surviving box

    "None" in the output shapes just means the count depends on how many boxes
    pass the threshold -- it isn't fixed ahead of time.
    """

    # combine "is there an object" confidence with per-class probability
    box_scores = box_class_probs * box_confidence

    # for each box, take the winning class and its score
    box_classes = tf.math.argmax(box_scores, axis=-1)
    box_class_scores = tf.math.reduce_max(box_scores, axis=-1)

    # keep only boxes whose winning score clears the threshold
    filtering_mask = box_class_scores >= threshold

    scores = tf.boolean_mask(box_class_scores, filtering_mask)
    boxes = tf.boolean_mask(boxes, filtering_mask)
    classes = tf.boolean_mask(box_classes, filtering_mask)

    return scores, boxes, classes

In [ ]:
# quick check against random tensors just to confirm shapes/types line up
tf.random.set_seed(10)
box_confidence = tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1)
boxes = tf.random.normal([19, 19, 5, 4], mean=1, stddev=4, seed = 1)
box_class_probs = tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1)
scores, boxes, classes = yolo_filter_boxes(boxes, box_confidence, box_class_probs, threshold = 0.5)

print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.shape))
print("boxes.shape = " + str(boxes.shape))
print("classes.shape = " + str(classes.shape))

Sample output on the check above (random inputs, so only the shapes matter here):

- `scores.shape` → (1789,)
- `boxes.shape` → (1789, 4)
- `classes.shape` → (1789,)

Note the inputs above are just random noise for a quick shape/type check — with real predictions, `box_class_probs` would hold genuine probabilities between 0 and 1, and box coordinates would always describe valid (non-negative) widths and heights.

<a name='2-3'></a>
### 2.3 - Non-Max Suppression

Thresholding on score alone still leaves plenty of overlapping boxes — the next filter, non-max suppression (NMS), handles that. *(Illustration of duplicate detections on the same car, collapsed down to one box, omitted here — image asset not included.)*

NMS leans on a function called **Intersection over Union (IoU)** — essentially, how much do two boxes overlap relative to their combined area.

**Convention used here:** origin (0,0) is the top-left of the image; x grows rightward, y grows downward. Boxes are defined by their corners, $(x_1, y_1)$ top-left and $(x_2, y_2)$ bottom-right, which makes the intersection math straightforward:

- Intersection top-left: the *larger* of the two boxes' $x_1$ / $y_1$ values
- Intersection bottom-right: the *smaller* of the two boxes' $x_2$ / $y_2$ values
- If the resulting width or height comes out negative, the boxes don't actually overlap — intersection area is 0
- Boxes that only touch at an edge or a corner also have zero overlap area

`iou()` below implements exactly that, plus the union area via $Union(A,B) = A + B - Intersection(A,B)$.

In [ ]:
def iou(box1, box2):
    """Computes Intersection over Union between two boxes.

    Arguments:
    box1 -- (box1_x1, box1_y1, box1_x2, box1_y2)
    box2 -- (box2_x1, box2_y1, box2_x2, box2_y2)
    """

    (box1_x1, box1_y1, box1_x2, box1_y2) = box1
    (box2_x1, box2_y1, box2_x2, box2_y2) = box2

    # corners of the overlapping region
    xi1 = max(box1_x1, box2_x1)
    yi1 = max(box1_y1, box2_y1)
    xi2 = min(box1_x2, box2_x2)
    yi2 = min(box1_y2, box2_y2)
    inter_width = xi2 - xi1
    inter_height = yi2 - yi1
    inter_area = max(inter_width, 0) * max(inter_height, 0)

    # union = sum of both areas minus the overlap counted twice
    box1_area = (box1_x2 - box1_x1) * (box1_y2 - box1_y1)
    box2_area = (box2_x2 - box2_x1) * (box2_y2 - box2_y1)
    union_area = box1_area + box2_area - inter_area

    iou = inter_area / union_area

    return iou

In [ ]:
# a few hand-picked cases: overlapping, disjoint, touching at a vertex, touching at an edge
box1 = (2, 1, 4, 3)
box2 = (1, 2, 3, 4)

print("iou for intersecting boxes = " + str(iou(box1, box2)))
assert iou(box1, box2) < 1, "The intersection area must be always smaller or equal than the union area."
assert np.isclose(iou(box1, box2), 0.14285714), "Wrong value. Check your implementation. Problem with intersecting boxes"

box1 = (1,2,3,4)
box2 = (5,6,7,8)
print("iou for non-intersecting boxes = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection must be 0"

box1 = (1,1,2,2)
box2 = (2,2,3,3)
print("iou for boxes that only touch at vertices = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection at vertices must be 0"

box1 = (1,1,3,3)
box2 = (2,3,3,4)
print("iou for boxes that only touch at edges = " + str(iou(box1,box2)))
assert iou(box1, box2) == 0, "Intersection at edges must be 0"

print("All tests passed!")

Expected values from the checks above: `0.1428...` for the overlapping pair, and `0.0` for the disjoint, vertex-touching, and edge-touching pairs — matches.

<a name='2-4'></a>
### 2.4 - Applying NMS

With IoU in hand, actual non-max suppression is:

1. Take the box with the highest score.
2. Compare it (via IoU) against every other box of the *same class* — drop any that overlap it past `iou_threshold`.
3. Repeat with whatever's left, until nothing above the threshold remains.

`yolo_non_max_suppression` below runs this per class (so a car overlapping a pedestrian never gets suppressed against each other), using TensorFlow's built-in `tf.image.non_max_suppression` to do the heavy lifting per class, then re-assembles and sorts the result.

In [ ]:
def yolo_non_max_suppression(scores, boxes, classes, max_boxes = 10, iou_threshold = 0.5):
    """
    Runs non-max suppression on a set of scored boxes, per class.

    Arguments:
    scores -- tensor (None,), from yolo_filter_boxes()
    boxes -- tensor (None, 4), from yolo_filter_boxes(), scaled to image size
    classes -- tensor (None,), from yolo_filter_boxes()
    max_boxes -- cap on how many boxes to keep overall
    iou_threshold -- overlap threshold above which a box is suppressed

    Returns:
    scores, boxes, classes -- same shapes as input, filtered down and sorted by score
    """
    boxes = tf.cast(boxes, dtype=tf.float32)
    scores = tf.cast(scores, dtype=tf.float32)

    nms_indices = []
    classes_labels = tf.unique(classes)[0]  # every class that appears at least once

    for label in classes_labels:
        filtering_mask = classes == label

        # isolate just this class's boxes/scores
        boxes_label = tf.boolean_mask(boxes, filtering_mask)
        scores_label = tf.boolean_mask(scores, filtering_mask)

        if tf.shape(scores_label)[0] > 0:
            # NMS within this class only
            nms_indices_label = tf.image.non_max_suppression(
                    boxes_label,
                    scores_label,
                    max_boxes,
                    iou_threshold=iou_threshold)

            # map back to indices in the original (pre-class-split) arrays
            selected_indices = tf.squeeze(tf.where(filtering_mask), axis=1)
            nms_indices.append(tf.gather(selected_indices, nms_indices_label))

    # merge every class's surviving indices back into one list
    nms_indices = tf.concat(nms_indices, axis = 0)

    scores = tf.gather(scores, nms_indices)
    boxes = tf.gather(boxes, nms_indices)
    classes = tf.gather(classes, nms_indices)

    # keep only the top max_boxes overall, highest score first
    sort_order = tf.argsort(scores, direction='DESCENDING').numpy()
    scores = tf.gather(scores, sort_order[0:max_boxes])
    boxes = tf.gather(boxes, sort_order[0:max_boxes])
    classes = tf.gather(classes, sort_order[0:max_boxes])

    return scores, boxes, classes

In [ ]:
# mimics a car and a person overlapping heavily -- different classes, so neither should be suppressed
scores = np.array([0.855, 0.828])
boxes = np.array([[0.45, 0.2,  1.01, 2.6], [0.42, 0.15, 1.7, 1.01]])
classes = np.array([0, 1])

print(f"iou:    \t{iou(boxes[0], boxes[1])}")

scores2, boxes2, classes2 = yolo_non_max_suppression(scores, boxes, classes, iou_threshold = 0.9)

assert np.allclose(scores2.numpy(), [0.855, 0.828])
assert np.allclose(boxes2.numpy(), boxes)
assert np.array_equal(classes2.numpy(), [0, 1])
print("scores2 = " + str(scores2.numpy()))
print("boxes2 = " + str(boxes2.numpy()))
print("classes2 = " + str(classes2.numpy()))
print("Test passed!")

As expected: the IoU between the two boxes is fairly high (~0.23), but since they belong to different classes, both boxes survive NMS untouched.

<a name='2-5'></a>
### 2.5 - Putting It Together

Last piece: a single function that takes the CNN's raw (19,19,5,85) encoding and runs it through score filtering *and* NMS to produce the final predictions.

One extra detail: YOLO internally represents boxes as (midpoint, height, width), so before filtering, boxes need converting to (top-left corner, bottom-right corner) format — that's what `yolo_boxes_to_corners` does below.

In [ ]:
def yolo_boxes_to_corners(box_xy, box_wh):
    """Converts (center, width/height) box format into (corner, corner) format."""
    box_mins = box_xy - (box_wh / 2.)
    box_maxes = box_xy + (box_wh / 2.)

    return tf.keras.backend.concatenate([
        box_mins[..., 1:2],  # y_min
        box_mins[..., 0:1],  # x_min
        box_maxes[..., 1:2],  # y_max
        box_maxes[..., 0:1]  # x_max
    ])

In [ ]:
def yolo_eval(yolo_outputs, image_shape = (720, 1280), max_boxes=10, score_threshold=.6, iou_threshold=.5):
    """
    Full pipeline: takes the raw YOLO encoding and returns the final filtered
    boxes, scores, and classes.

    Arguments:
    yolo_outputs -- 4 tensors from the model:
                    box_xy: (None, 19, 19, 5, 2)
                    box_wh: (None, 19, 19, 5, 2)
                    box_confidence: (None, 19, 19, 5, 1)
                    box_class_probs: (None, 19, 19, 5, 80)
    image_shape -- shape to scale the final boxes back to (float32)
    max_boxes -- cap on returned boxes
    score_threshold -- min class score to keep a box
    iou_threshold -- overlap threshold for NMS

    Returns:
    scores, boxes, classes -- final predictions
    """

    # unpack the model's raw output
    box_xy, box_wh, box_confidence, box_class_probs = yolo_outputs

    # convert to corner format so filtering functions can use it
    boxes = yolo_boxes_to_corners(box_xy, box_wh)

    # step 1: drop low-confidence boxes
    scores, boxes, classes = yolo_filter_boxes(boxes,
                                  box_confidence,
                                  box_class_probs,
                                  score_threshold
                                 )

    # rescale from the model's 608x608 space back to the original image size
    boxes = scale_boxes(boxes, image_shape)

    # step 2: collapse overlapping duplicates via NMS
    scores, boxes, classes = yolo_non_max_suppression(scores,
                                  boxes,
                                  classes,
                                  max_boxes,
                                  iou_threshold = iou_threshold
                                 )

    return scores, boxes, classes

In [ ]:
# end-to-end check with random tensors standing in for a real model's output
tf.random.set_seed(10)
yolo_outputs = (tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 2], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 1], mean=1, stddev=4, seed = 1),
                tf.random.normal([19, 19, 5, 80], mean=1, stddev=4, seed = 1))
scores, boxes, classes = yolo_eval(yolo_outputs)
print("scores[2] = " + str(scores[2].numpy()))
print("boxes[2] = " + str(boxes[2].numpy()))
print("classes[2] = " + str(classes[2].numpy()))
print("scores.shape = " + str(scores.numpy().shape))
print("boxes.shape = " + str(boxes.numpy().shape))
print("classes.shape = " + str(classes.numpy().shape))

With `max_boxes=10` (the default), the pipeline correctly caps out at 10 returned boxes regardless of how many candidates went in — shapes come out as `(10,)`, `(10, 4)`, `(10,)`.

<a name='3'></a>
## 3 - Running the Pre-trained Model

With the filtering pipeline done, the next step is pointing it at an actual trained YOLO model and a real image.

<a name='3-1'></a>
### 3.1 - Classes, Anchors, Image Shape

80 target classes, 5 anchor boxes — both are read in from text files (`coco_classes.txt`, `yolo_anchors.txt`). Source images are 720×1280 and get resized to 608×608 before going into the model.

In [ ]:
class_names = read_classes("model_data/coco_classes.txt")
anchors = read_anchors("model_data/yolo_anchors.txt")
model_image_size = (608, 608) # matches the model's input layer

<a name='3-2'></a>
### 3.2 - Loading the Model

Training YOLO from scratch needs a large labeled dataset and a lot of compute, so this uses pre-trained weights instead — originally from the official YOLO website, converted to a Keras-loadable format by Allan Zelener's YAD2K project (full credit in the References section). These are technically YOLOv2 weights, referred to here simply as "YOLO" for consistency with the rest of the notebook.

In [ ]:
yolo_model = load_model("model_data/", compile=False)

Layer summary of the loaded model:

In [ ]:
yolo_model.summary()

(A Keras warning may show up here on some setups — safe to ignore.)

This model takes a preprocessed batch of images, shape (m, 608, 608, 3), and outputs the raw encoding, shape (m, 19, 19, 5, 85).

<a name='3-3'></a>
### 3.3 - Formatting the Model Output

The model's raw (m, 19, 19, 5, 85) output isn't directly usable yet — `yolo_head` reshapes it into the four separate tensors (`box_xy`, `box_wh`, `box_confidence`, `box_class_probs`) that `yolo_eval` expects:

```
yolo_model_outputs = yolo_model(image_data)
yolo_outputs = yolo_head(yolo_model_outputs, anchors, len(class_names))
```

`yolo_head`'s implementation lives in `yad2k/models/keras_yolo.py` if you want to see exactly how the reshaping works.

<a name='3-4'></a>
### 3.4 - Filtering the Boxes

`yolo_outputs` now holds every candidate box in a usable format — `yolo_eval` (built above) runs the actual filtering:

```
out_scores, out_boxes, out_classes = yolo_eval(yolo_outputs, [image.size[1], image.size[0]], 10, 0.3, 0.5)
```

<a name='3-5'></a>
### 3.5 - Running Inference

Full chain, start to finish:

1. Feed a preprocessed image into `yolo_model` → raw output
2. Reshape via `yolo_head` → `yolo_outputs`
3. Filter via `yolo_eval` → final `out_scores`, `out_boxes`, `out_classes`

`predict()` below wraps that whole chain into one call, plus preprocessing and drawing.

It relies on `preprocess_image("images/" + image_file, model_image_size=(608, 608))`, which loads, resizes, and normalizes the image, returning:
- `image` — the PIL image (used for drawing the final boxes)
- `image_data` — the corresponding numpy array fed into the CNN

In [ ]:
def predict(image_file):
    """
    Runs the full detection pipeline on an image and displays the result.

    Arguments:
    image_file -- filename of an image inside the "images" folder

    Returns:
    out_scores, out_boxes, out_classes -- final predictions (count varies, capped at max_boxes)
    """

    image, image_data = preprocess_image("images/" + image_file, model_image_size = (608, 608))

    yolo_model_outputs = yolo_model(image_data)
    yolo_outputs = yolo_head(yolo_model_outputs, anchors, len(class_names))

    out_scores, out_boxes, out_classes = yolo_eval(yolo_outputs, [image.size[1],  image.size[0]], 10, 0.3, 0.5)

    print('Found {} boxes for {}'.format(len(out_boxes), "images/" + image_file))
    colors = get_colors_for_classes(len(class_names))
    draw_boxes(image, out_boxes, out_classes, class_names, out_scores)
    image.save(os.path.join("out", image_file), quality=100)
    output_image = Image.open(os.path.join("out", image_file))
    imshow(output_image)

    return out_scores, out_boxes, out_classes

Running it on `test.jpg` to check everything works end to end:

In [ ]:
out_scores, out_boxes, out_classes = predict("test.jpg")

On this image the pipeline finds 10 boxes — mostly cars, plus a bus and a traffic light, each with a confidence score and corner coordinates. Scores here range from ~0.89 down to ~0.36, which lines up with what you'd expect: closer, clearer vehicles score higher than a small traffic light in the background.

The loaded model can recognize all 80 COCO classes (full list in `coco_classes.txt`), not just cars. To try it on a different image: drop the file into the `images/` folder, update the filename passed to `predict()`, and rerun.

*(The original assignment includes a compiled video of predictions run across a whole driving sequence — omitted here since the video file isn't part of this repo. Credit to drive.ai for the underlying dataset those frames came from.)*

<a name='4'></a>
## 4 - Recap

- Input image: (608, 608, 3)
- CNN output: (19, 19, 5, 85), which flattens to (19, 19, 425)
    - 5 boxes per grid cell (one per anchor box)
    - Each box: 85 numbers = 5 for $(p_c, b_x, b_y, b_h, b_w)$ + 80 class probabilities
- Final predictions come from two filtering steps:
    - **Score thresholding** — drop boxes below a minimum class-confidence score
    - **Non-max suppression** — use IoU to collapse duplicate/overlapping detections into one box per object

**Key takeaways**

- YOLO gets both speed and accuracy by making all predictions in a single forward pass
- The output encoding is a 19×19 grid, 5 boxes per cell, 85 numbers per box
- Score thresholding + IoU-based non-max suppression together turn hundreds of raw candidate boxes into a small, clean set of final detections
- Training YOLO from scratch is expensive (data + compute), which is why this notebook uses pre-trained weights rather than training from zero — fine-tuning on a custom dataset is a natural next step from here

This covers the full pipeline: score filtering, IoU, non-max suppression, and running a pre-trained YOLO model end-to-end on a real image. See below for everything this project builds on.

<a name='5'></a>
## 5 - References & Credits

This project was completed as part of DeepLearning.AI's Deep Learning Specialization (Convolutional Neural Networks course), taught by Andrew Ng — the course structure and lecture content are what taught me how YOLO's filtering pipeline works, and the assignment this is based on is where I built and tested this implementation. The object-detection logic in this notebook (score filtering, IoU, NMS) is my own code, written to implement the algorithm as described in the papers below.

- Joseph Redmon, Santosh Divvala, Ross Girshick, Ali Farhadi — [You Only Look Once: Unified, Real-Time Object Detection](https://arxiv.org/abs/1506.02640) (2015)
- Joseph Redmon, Ali Farhadi — [YOLO9000: Better, Faster, Stronger](https://arxiv.org/abs/1612.08242) (2016)
- Allan Zelener — [YAD2K: Yet Another Darknet 2 Keras](https://github.com/allanzelener/YAD2K) (pre-trained weight conversion)
- Official YOLO website: https://pjreddie.com/darknet/yolo/

### Dataset

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br /><span xmlns:dct="http://purl.org/dc/terms/" property="dct:title">The Drive.ai Sample Dataset</span> is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Thanks to Brody Huval, Chih Hu, and Rahul Patel for providing this data.